## DataFoundry Entity: history tab (interface)
### GOAL: Get temperature dataset for the last 3-5 days and upload to DataFoundry entity dataset

In [646]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

### Import csv files from DataFoundry

In [803]:
# Get the database from the DataFoundry link
df_window = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/QWM2TVQ4OE9KRWF1UU1tRUxzbXJFS1IxM3hhMVZORnNlSlMvRmVyZUFsdz0=", low_memory=False)
df_window = df_window[(df_window.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_window = df_window.drop(["Unnamed: 7", "device_id", "id", "participant", "recipient", "pp1", "pp2", "pp3", "activity", "light", "participant", "curtain"], axis='columns')

# get the left window sensor values
df_window_left = df_window.loc[(df_window['sender'] == "HH2_window_1") | (df_window['sender'] == "HH2_window_1_V2")].copy()
df_window_left["curtain_left"] = np.where(df_window_left.loc[:,"distance sensor"] > 50, 1, 0) # set curtain state (0= closed; 1 = open)
# df_window_left = df_window_left.rename(columns={"light sensor": "light_left"}, errors="raise")
df_window_left["shade_left"] = np.where(df_window_left.loc[:,"light sensor"] < 2000, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_left = df_window_left.drop(["sender", "window", "distance sensor", "reed sensor", "light sensor"], axis='columns')
df_window_left['ts'] = pd.to_datetime(df_window_left['ts']) #Turn timestamp into datetime dtype
df_window_left = df_window_left.drop_duplicates() #Drop duplicate rows
# df_window_left.resample('10min', on='ts').last() #get one value per 10 minutes
df_window_left['hr'] = df_window_left['ts'].dt.hour
df_window_left['date'] = df_window_left['ts'].dt.date
df_window_left = df_window_left.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
df_window_left['ts'] = df_window_left["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_left = df_window_left.drop(["hr", "date"], axis='columns')

# # get the right window sensor values
df_window_right = df_window.loc[df_window['sender'] == "HH2_window_2"].copy()
df_window_right["curtain_right"] = np.where(df_window_right.loc[:,"distance sensor"] > 50, 1, 0) # set curtain state (0= closed; 1 = open)
# df_window_right = df_window_right.rename(columns={"distance sensor": "distance_right"}, errors="raise")
# df_window_right = df_window_right.rename(columns={"light sensor": "light_right"}, errors="raise")
df_window_right["shade_right"] = np.where(df_window_right.loc[:,"light sensor"] < 2000, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_right = df_window_right.drop(["sender", "window", "distance sensor", "reed sensor", "light sensor"], axis='columns')
df_window_right['ts'] = pd.to_datetime(df_window_right['ts']) #Turn timestamp into datetime dtype
df_window_right = df_window_right.drop_duplicates() #Drop duplicate rows
# df_window_right.resample('10min', on='ts').last() #get one value per 10 minutes
# df_window_right['ts'] = df_window_right["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_right['hr'] = df_window_right['ts'].dt.hour
df_window_right['date'] = df_window_right['ts'].dt.date
df_window_right = df_window_right.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
df_window_right['ts'] = df_window_right["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_right = df_window_right.drop(["hr", "date"], axis='columns')


# # get the right window sensor values
df_window_door = df_window.loc[df_window['sender'] == "HH2_windowdoor_1"].copy()
df_window_door = df_window_door.rename(columns={"reed sensor": "window_door"}, errors="raise")
df_window_door = df_window_door.drop(["sender", "light sensor", "window", "distance sensor"], axis='columns')
df_window_door['ts'] = pd.to_datetime(df_window_door['ts']) #Turn timestamp into datetime dtype
df_window_door = df_window_door.drop_duplicates() #Drop duplicate rows
# df_window_door.resample('10min', on='ts').last() #get one value per 10 minutes
df_window_door['hr'] = df_window_door['ts'].dt.hour
df_window_door['date'] = df_window_door['ts'].dt.date
df_window_door = df_window_door.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
df_window_door['ts'] = df_window_door["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_door = df_window_door.drop(["hr", "date"], axis='columns')

#### Create a df with a timestamp for each hour of the last three days -- merge the dataframes with this df (makes sure there's values for each hour + limits the df to the last 72 hours)
# Get the current time
now = datetime.now()  + timedelta(hours=0) #add two hours to the current time since we are in CST+2 (and pythonanywhere in CST)
# today = now.strftime('%Y-%m-%d') + "T00:00:00"
# Create a date range for the last 24 hours with hourly frequency
last_day_range = pd.date_range(end=now, periods=300, freq='H')
# Create the DataFrame
last_days = pd.DataFrame(last_day_range, columns=['ts'])
last_days['ts'] = last_days["ts"].dt.round('h')  #Round the datestamp column to hours

### merge the window dataframes on the timestamps for the last 72 hours
from functools import reduce
# Create a list of DataFrames to merge
dataframes_to_merge = [
    last_days,
    df_window_left,
    df_window_right,
    df_window_door
]

# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Use reduce to merge all DataFrames in one go
df_window = reduce(merge_asof, dataframes_to_merge)

# df_window = df_window.drop_duplicates(subset=['ts']) #Drop duplicate rows

df_window.tail()

,ts,curtain_left,shade_left,curtain_right,shade_right,window_door
295,2024-10-04 16:00:00,1.0,0.0,1,0,0.0
296,2024-10-04 17:00:00,1.0,0.0,1,0,0.0
297,2024-10-04 18:00:00,1.0,0.0,1,0,0.0
298,2024-10-04 19:00:00,1.0,1.0,1,1,0.0
299,2024-10-04 20:00:00,1.0,1.0,1,1,0.0


In [804]:
# Get the database from the DataFoundry link
df_door = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/S0lMV3BUZ3NjZ2lQVUJUVHlJREhSM2NxNkFGT0VscHFHY0xjdXZVTS9SST0=")
df_door = df_door[(df_door.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_door = df_door.drop(["window", "Unnamed: 7", "device_id", "activity", "participant", "sender", "id", "recipient", "pp1", "pp2", "pp3"], axis='columns')
df_door = df_door.rename(columns={"reed sensor": "door_indoors"}, errors="raise")
df_door['ts'] = pd.to_datetime(df_door['ts']) ## Turn timestamp into datetime dtype
df_door = df_door.dropna() ## drop rows with empty (NA) cells
df_door = df_door.drop_duplicates() ## Drop duplicate rows
df_door['hr'] = df_door['ts'].dt.hour
df_door['date'] = df_door['ts'].dt.date
df_door = df_door.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
# df_door = df_door.groupby(['date', 'hr']).agg(pd.Series.mode).reset_index()
df_door['ts'] = df_door["ts"].dt.round('h')  #Round the datestamp column to hours
df_door = df_door.drop(["hr", "date"], axis='columns')
df_door.tail()

,ts,door_indoors
891,2024-10-04 16:00:00,1.0
892,2024-10-04 17:00:00,1.0
893,2024-10-04 18:00:00,1.0
894,2024-10-04 19:00:00,0.0
895,2024-10-04 20:00:00,0.0


In [805]:
# Get the database from the DataFoundry link
df_indoor = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/Yy9PQlp3clNidmNyc0pYS1JBV1NlQ1JpbGNKWHBIQlVVMjlwQW9nOFY5UT0=", low_memory=False)
df_indoor = df_indoor[(df_indoor.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_indoor = df_indoor.drop(["Unnamed: 7", "device_id", "activity", "participant", "sender", "id", "recipient", "pp1", "pp2", "pp3"], axis='columns')
df_indoor = df_indoor.rename(columns={"Temperature": "Temperature_indoor"}, errors="raise")
df_indoor['ts'] = pd.to_datetime(df_indoor['ts']) ## Turn timestamp into datetime dtype
df_indoor['Temperature_indoor'] = np.round(df_indoor['Temperature_indoor'] * 10) / 10 ## round temperature to 1 decimal
df_indoor = df_indoor.dropna() ## drop rows with empty (NA) cells
df_indoor = df_indoor.drop_duplicates() ## Drop duplicate rowsindex_list= df_indoor2.Timestamp[(df_indoor2.Timestamp >= "2024-08-08 16:00:00") & (df_indoor2.Timestamp <= "2024-08-08 19:20:00")].index.tolist(
df_indoor['hr'] = df_indoor['ts'].dt.hour
df_indoor['date'] = df_indoor['ts'].dt.date
df_indoor = df_indoor.groupby(['date', 'hr']).first().reset_index()
df_indoor['ts'] = df_indoor["ts"].dt.round('h')  #Round the datestamp column to hours
df_indoor = df_indoor.drop(["hr", "date"], axis='columns')

In [806]:
df_indoor.tail()

,ts,Temperature_indoor
892,2024-10-04 15:00:00,23.1
893,2024-10-04 16:00:00,22.5
894,2024-10-04 17:00:00,22.1
895,2024-10-04 18:00:00,21.5
896,2024-10-04 19:00:00,21.5


In [807]:
# Get the database from the DataFoundry link
df_API = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/N0FsN2loZlVYMWhBaE0rd0l5T2NadzR3YTNBcnovQlJ0SG13dHMxL0U1RT0=")
df_API = df_API[(df_API.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_API = df_API.drop(["activity", "device_id", "sender", "participant", "Unnamed: 7", "id", "recipient", "pp1", "pp2", "pp3", "Temperature_API_MAX", "Temperature_API_MIN", "weather_description", "weather_main"], axis='columns')
df_API['ts'] = pd.to_datetime(df_API['ts']) ## Turn timestamp into datetime dtype
df_API['Temperature_API'] = np.round(df_API['Temperature_API'] * 10) / 10 ## round temperature to 1 decimal
df_API['hr'] = df_API['ts'].dt.hour
df_API['date'] = df_API['ts'].dt.date
df_API = df_API.groupby(['date', 'hr']).first().reset_index()
df_API['ts'] = df_API["ts"].dt.round('h')  #Round the datestamp column to hours
df_API = df_API.drop(["hr", "date"], axis='columns')

### Export json to entity dataset in DataFoundry (last5days)

In [809]:
# Create a list of DataFrames to merge
dataframes_to_merge = [
    df_window,
    df_indoor,
    df_door,
    df_API
]
# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))
# Use reduce to merge all DataFrames in one go
df_merge = reduce(merge_asof, dataframes_to_merge)
# get a column with only the day and month (lvgl labeling purposes)
df_merge['date'] = df_merge['ts'].dt.strftime('%d/%m') 


## get difference between today midnight and the timestamp for each row
today = datetime.now()
today_morning = today.strftime('%Y-%m-%d') + "T00:00:00"
df_merge["time_since_today"] = today_morning
df_merge['time_since_today'] = pd.to_datetime(df_merge['time_since_today']) ## Turn timestamp into datetime dtype
df_merge['time_since_today'] = (df_merge.ts - df_merge.time_since_today) / pd.Timedelta(hours=1)
df_merge = df_merge.drop_duplicates(subset=["ts"]) ## Drop duplicate rows
# # get rows for the last three days only, and reset the index
# df_merge = df_merge[(df_merge.time_since_today > -73) & (df_merge.time_since_today < 0)]
# df_merge = df_merge.reset_index(drop=True)

# Get the min and max values for 'Temperature_indoor' and 'Temperature_API' to set the range in the lvgl chart
minvalue = df_merge[["Temperature_indoor", "Temperature_API"]].min().min()  # Overall minimum value
maxvalue = df_merge[["Temperature_indoor", "Temperature_API"]].max().max()  # Overall maximum value

# Assign the min and max values to new columns 'pp1' and 'pp2'
df_merge["pp1"] = minvalue
df_merge["pp2"] = maxvalue

# Convert specified columns to strings, as only strings can be passed as JSON parameters
columns_to_convert = ['pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'time_since_today', 'Temperature_API', 'Temperature_indoor']
df_merge[columns_to_convert] = df_merge[columns_to_convert].astype('object')
df_merge['ts'] = df_merge['ts'].map(str)
df_merge['date'] = df_merge['date'].astype('object')

df_merge.tail()

,ts,curtain_left,shade_left,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,Temperature_API,date,time_since_today,pp1,pp2
295,2024-10-04 16:00:00,1.0,0.0,1,0,0.0,22.5,1.0,14.8,04/10,16.0,3.6,26.2
296,2024-10-04 17:00:00,1.0,0.0,1,0,0.0,22.1,1.0,14.8,04/10,17.0,3.6,26.2
297,2024-10-04 18:00:00,1.0,0.0,1,0,0.0,21.5,1.0,14.4,04/10,18.0,3.6,26.2
298,2024-10-04 19:00:00,1.0,1.0,1,1,0.0,21.5,0.0,13.3,04/10,19.0,3.6,26.2
299,2024-10-04 20:00:00,1.0,1.0,1,1,0.0,21.5,0.0,13.3,04/10,20.0,3.6,26.2


In [810]:
df_merge = df_merge.reindex(index=df_merge.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
#get a string id per row to pass as unique resource_id
# Reset index and create a unique 'id' column
df_merge.reset_index(inplace=True, drop=True)
df_merge["id"] = df_merge.index + 1
df_merge['id'] = df_merge['id'].map(str)


# Get rows for the last three days only, and reset the index for each subset
df_yesterday = df_merge[(df_merge.time_since_today > -25) & (df_merge.time_since_today < 0)]
df_yesterday = df_yesterday.reset_index(drop=True)  # Reset index for yesterday's data

df_twodaysago = df_merge[(df_merge.time_since_today > -49) & (df_merge.time_since_today < -24)]
df_twodaysago = df_twodaysago.reset_index(drop=True)  # Reset index for two days ago's data

df_threedaysago = df_merge[(df_merge.time_since_today > -73) & (df_merge.time_since_today < -48)]
df_threedaysago = df_threedaysago.reset_index(drop=True)  # Reset index for three days ago's data

# Make sure again that all columns are of object dtype
columns_to_convert = ['ts', 'date','pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'time_since_today', 'Temperature_API', 'Temperature_indoor']
df_yesterday[columns_to_convert] = df_yesterday[columns_to_convert].astype('object')
df_twodaysago[columns_to_convert] = df_twodaysago[columns_to_convert].astype('object')
df_threedaysago[columns_to_convert] = df_threedaysago[columns_to_convert].astype('object')

# Fill NaN values with 0 (since they will mess up the JSON output)
df_yesterday.fillna("0", inplace=True)
df_twodaysago.fillna("0", inplace=True)
df_threedaysago.fillna("0", inplace=True)

# Today and tomorrow

In [812]:
#### Create a df for today
# Get the current time
now = datetime.now()
today_morning = now.strftime('%Y-%m-%d') + "T00:00:00"
today_range = pd.date_range(start=today_morning, periods=24, freq='H') # Create a date range for the last 24 hours with hourly frequency
today_full = pd.DataFrame(today_range, columns=['ts']) # Create the DataFrame

df_today = df_merge[(df_merge.time_since_today >= 0) & (df_merge.time_since_today < 24)].copy() # get the values that have been measured up until the current time today
df_today['ts'] = pd.to_datetime(df_today['ts'])  # Convert 'ts' to datetime dtype

# Create a list of DataFrames to merge
dataframes_to_merge = [
    today_full,
    df_today]
# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('50min'))

# Use reduce to merge all DataFrames in one go
df_today = reduce(merge_asof, dataframes_to_merge)
df_today = df_today.reset_index(drop=True)  # Reset index for the merged data
df_today['ts'] = df_today['ts'].map(str) # make sure ts is a string type again (for json)
df_today = df_today.reindex(index=df_today.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
df_today.reset_index(inplace=True, drop=True)
df_today["id"] = df_today.index + 1
df_today['id'] = df_today['id'].map(str)

In [813]:
#json
import requests

# api-endpoint
URL = "https://pro.openweathermap.org/data/2.5/forecast/hourly?lat=51.3135296&lon=4.83753463479851&appid=273a7bdc5f26d184400ce98d1bc8e957&units=metric"
 
# Sending a GET request to the specified URL to retrieve weather data
response = requests.get(url = URL)

# Extracting data from the response in JSON format
data = response.json()

# Flatten (normalize) the JSON file to create a DataFrame from the 'list' key
list = pd.json_normalize(data, record_path=['list'])  # Extract main data (list)
weather = pd.json_normalize(data, record_path=['list', 'weather'])  # Extract weather descriptions into a separate DataFrame

# Combining the main data and weather DataFrame, and dropping unnecessary columns
forecast = pd.concat([list, weather], axis=1)
forecast = forecast.drop(["main.temp_min", "main.temp_max", "visibility", 'description', "weather", 'main', "dt", "id", "pop", "main.feels_like", "main.pressure", "clouds.all", "wind.speed", "wind.deg", "wind.gust", "sys.pod", "rain.1h", "id", "icon", "main.sea_level", "main.grnd_level", "main.humidity", "main.temp_kf"], axis='columns')

# Renaming columns for better readability and consistency
forecast = forecast.rename(columns={"dt_txt": "ts"}, errors="raise")
forecast = forecast.rename(columns={"main.temp": "Temperature_API"}, errors="raise")

# Converting the 'ts' column to datetime format
forecast['ts'] = pd.to_datetime(forecast['ts'])  # Turn timestamp into datetime dtype

# Getting the current time and adding one hour to it
start_time = datetime.now() + timedelta(hours=0)  # Set the start time to the next hour from the current time

# Filtering the forecast DataFrame to include only rows where the timestamp is greater than or equal to the start time
forecast = forecast[forecast['ts'] >= start_time]  # Keep only future forecasts starting from one hour from now

# Rounding temperature values to the nearest whole number
forecast = forecast.round(0)  # Round temperature to 0 decimal places

# Convert the string timestamps to datetime objects for comparison
start_time = pd.to_datetime('2024-10-04 21:00:00')
end_time = pd.to_datetime('2024-10-04 23:00:00')

# Filtering the forecast DataFrame to include only rows where the timestamp is between the two datetime objects
forecast_today = forecast[(forecast['ts'] >= start_time) & (forecast['ts'] <= end_time)]

# Ensure both 'ts' columns are of datetime type before merging
df_today['ts'] = pd.to_datetime(df_today['ts'])  # Convert 'ts' in df_today to datetime

# Use .loc to avoid SettingWithCopyWarning when modifying a slice of the DataFrame
forecast_today.loc[:, 'ts'] = pd.to_datetime(forecast_today['ts'])  # Convert 'ts' in forecast_filtered to datetime

In [814]:
# Merge the two DataFrames on the 'ts' column using an asof merge
df_today_max = pd.merge_asof(df_today.sort_values('ts'), forecast_today.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Combine the Temperature_API values, filling NaN values in df_today with values from forecast_filtered
df_today_max['Temperature_API'] = df_today_max['Temperature_API_x'].combine_first(df_today_max['Temperature_API_y'])
# Drop the original Temperature_API_x and Temperature_API_y columns to avoid redundancy
df_today_max = df_today_max.drop(columns=['Temperature_API_x', 'Temperature_API_y'])

# Manually enter values for the last three rows of specified columns
df_today_max.loc[df_today_max.index[-3:], 'Temperature_indoor'] = [21, 20, 20]  # Replace value1, value2, value3 with actual values
df_today_max.loc[df_today_max.index[-3:], 'door_indoors'] = [0.0, 0.0, 0.0]  # Replace value1, value2, value3 with actual values
df_today_max.loc[df_today_max.index[-3:], 'window_door'] = [0.0, 0.0, 0.0]  # Replace value1, value2, value3 with actual values
df_today_max.loc[df_today_max.index[-3:], 'curtain_right'] = [0.0, 0.0, 0.0]  # Replace value1, value2, value3 with actual values
df_today_max.loc[df_today_max.index[-3:], 'curtain_left'] = [0.0, 0.0, 0.0]  # Replace value1, value2, value3 with actual values
df_today_max.loc[df_today_max.index[-3:], 'shade_left'] = [1.0, 1.0, 1.0]  # Replace value1, value2, value3 with actual values
df_today_max.loc[df_today_max.index[-3:], 'shade_right'] = [1.0, 1.0, 1.0]  # Replace value1, value2, value3 with actual values

columns_to_convert = ['ts', 'date','pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'time_since_today', 'Temperature_API', 'Temperature_indoor']
df_today_max[columns_to_convert] = df_today_max[columns_to_convert].astype('object')
df_today_max['ts'] = df_today_max['ts'].map(str) # make sure ts is a string type again (for json)

# Fill NaN values with 0 (since they will mess up the JSON output)
df_today_max.fillna("0", inplace=True)
df_today_max = df_today_max.reindex(index=df_today_max.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
df_today_max.reset_index(inplace=True, drop=True)

df_today_max

,ts,curtain_left,shade_left,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,date,time_since_today,pp1,pp2,id,Temperature_API
0,2024-10-04 23:00:00,0.0,1.0,0.0,1.0,0.0,20,0.0,0,0,0,0,1,8.0
1,2024-10-04 22:00:00,0.0,1.0,0.0,1.0,0.0,20,0.0,0,0,0,0,2,9.0
2,2024-10-04 21:00:00,0.0,1.0,0.0,1.0,0.0,21,0.0,0,0,0,0,3,10.0
3,2024-10-04 20:00:00,1.0,1.0,1,1,0.0,21.5,0.0,04/10,20.0,3.6,26.2,4,13.3
4,2024-10-04 19:00:00,1.0,1.0,1,1,0.0,21.5,0.0,04/10,19.0,3.6,26.2,5,13.3
5,2024-10-04 18:00:00,1.0,0.0,1,0,0.0,21.5,1.0,04/10,18.0,3.6,26.2,6,14.4
6,2024-10-04 17:00:00,1.0,0.0,1,0,0.0,22.1,1.0,04/10,17.0,3.6,26.2,7,14.8
7,2024-10-04 16:00:00,1.0,0.0,1,0,0.0,22.5,1.0,04/10,16.0,3.6,26.2,8,14.8
8,2024-10-04 15:00:00,1.0,0.0,1,0,0.0,23.1,0.0,04/10,15.0,3.6,26.2,9,14.6
9,2024-10-04 14:00:00,1.0,0.0,1,0,0.0,22.4,0.0,04/10,14.0,3.6,26.2,10,14.4


In [815]:
# Merge the two DataFrames on the 'ts' column using an asof merge
df_today_min = pd.merge_asof(df_today.sort_values('ts'), forecast_filtered.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Combine the Temperature_API values, filling NaN values in df_today with values from forecast_filtered
df_today_min['Temperature_API'] = df_today_min['Temperature_API_x'].combine_first(df_today_min['Temperature_API_y'])
# Drop the original Temperature_API_x and Temperature_API_y columns to avoid redundancy
df_today_min = df_today_min.drop(columns=['Temperature_API_x', 'Temperature_API_y'])

# # Manually enter values for the last three rows of specified columns
df_today_min.loc[df_today_min.index[-3:], 'Temperature_indoor'] = [18, 16, 15]  # Replace value1, value2, value3 with actual values
df_today_min.loc[df_today_min.index[-3:], 'door_indoors'] = [1.0, 1.0, 0.0]  # Replace value1, value2, value3 with actual values
df_today_min.loc[df_today_min.index[-3:], 'window_door'] = [1.0, 1.0, 1.0]  # Replace value1, value2, value3 with actual values
df_today_min.loc[df_today_min.index[-3:], 'curtain_right'] = [1.0, 1.0, 1.0]  # Replace value1, value2, value3 with actual values
df_today_min.loc[df_today_min.index[-3:], 'curtain_left'] = [1.0, 1.0, 1.0]  # Replace value1, value2, value3 with actual values
df_today_min.loc[df_today_min.index[-3:], 'shade_left'] = [1.0, 1.0, 1.0]  # Replace value1, value2, value3 with actual values
df_today_min.loc[df_today_min.index[-3:], 'shade_right'] = [1.0, 1.0, 1.0]  # Replace value1, value2, value3 with actual values

columns_to_convert = ['ts', 'date','pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'time_since_today', 'Temperature_API', 'Temperature_indoor']
df_today_min[columns_to_convert] = df_today_min[columns_to_convert].astype('object')
df_today_min['ts'] = df_today_min['ts'].map(str) # make sure ts is a string type again (for json)

# Fill NaN values with 0 (since they will mess up the JSON output)
df_today_min.fillna("0", inplace=True)
df_today_min = df_today_min.reindex(index=df_today_min.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
df_today_min.reset_index(inplace=True, drop=True)

df_today_min

,ts,curtain_left,shade_left,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,date,time_since_today,pp1,pp2,id,Temperature_API
0,2024-10-04 23:00:00,1.0,1.0,1.0,1.0,1.0,15,0.0,0,0,0,0,1,8.0
1,2024-10-04 22:00:00,1.0,1.0,1.0,1.0,1.0,16,1.0,0,0,0,0,2,9.0
2,2024-10-04 21:00:00,1.0,1.0,1.0,1.0,1.0,18,1.0,0,0,0,0,3,9.0
3,2024-10-04 20:00:00,1.0,1.0,1,1,0.0,21.5,0.0,04/10,20.0,3.6,26.2,4,13.3
4,2024-10-04 19:00:00,1.0,1.0,1,1,0.0,21.5,0.0,04/10,19.0,3.6,26.2,5,13.3
5,2024-10-04 18:00:00,1.0,0.0,1,0,0.0,21.5,1.0,04/10,18.0,3.6,26.2,6,14.4
6,2024-10-04 17:00:00,1.0,0.0,1,0,0.0,22.1,1.0,04/10,17.0,3.6,26.2,7,14.8
7,2024-10-04 16:00:00,1.0,0.0,1,0,0.0,22.5,1.0,04/10,16.0,3.6,26.2,8,14.8
8,2024-10-04 15:00:00,1.0,0.0,1,0,0.0,23.1,0.0,04/10,15.0,3.6,26.2,9,14.6
9,2024-10-04 14:00:00,1.0,0.0,1,0,0.0,22.4,0.0,04/10,14.0,3.6,26.2,10,14.4


In [816]:
# neutral df for today
columns_to_convert = ['ts', 'date','pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'time_since_today', 'Temperature_API', 'Temperature_indoor']
df_today[columns_to_convert] = df_today[columns_to_convert].astype('object')
df_today['ts'] = df_today['ts'].map(str) # make sure ts is a string type again (for json)

# Fill NaN values with 0 (since they will mess up the JSON output)
df_today.fillna("0", inplace=True)
# df_today = df_today.reindex(index=df_today.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
# df_today.reset_index(inplace=True, drop=True)

df_today

,ts,curtain_left,shade_left,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,Temperature_API,date,time_since_today,pp1,pp2,id
0,2024-10-04 23:00:00,0,0,0,0,0,0,0,0,0,0,0,0,1
1,2024-10-04 22:00:00,0,0,0,0,0,0,0,0,0,0,0,0,2
2,2024-10-04 21:00:00,0,0,0,0,0,0,0,0,0,0,0,0,3
3,2024-10-04 20:00:00,1.0,1.0,1,1,0.0,21.5,0.0,13.3,04/10,20.0,3.6,26.2,4
4,2024-10-04 19:00:00,1.0,1.0,1,1,0.0,21.5,0.0,13.3,04/10,19.0,3.6,26.2,5
5,2024-10-04 18:00:00,1.0,0.0,1,0,0.0,21.5,1.0,14.4,04/10,18.0,3.6,26.2,6
6,2024-10-04 17:00:00,1.0,0.0,1,0,0.0,22.1,1.0,14.8,04/10,17.0,3.6,26.2,7
7,2024-10-04 16:00:00,1.0,0.0,1,0,0.0,22.5,1.0,14.8,04/10,16.0,3.6,26.2,8
8,2024-10-04 15:00:00,1.0,0.0,1,0,0.0,23.1,0.0,14.6,04/10,15.0,3.6,26.2,9
9,2024-10-04 14:00:00,1.0,0.0,1,0,0.0,22.4,0.0,14.4,04/10,14.0,3.6,26.2,10


## Tomorrow

In [749]:
# Convert the string timestamps to datetime objects for comparison
start_time = pd.to_datetime('2024-10-05 00:00:00')
end_time = pd.to_datetime('2024-10-05 23:00:00')

# Filtering the forecast DataFrame to include only rows where the timestamp is between the two datetime objects
forecast_tomorrow = forecast[(forecast['ts'] >= start_time) & (forecast['ts'] <= end_time)]

# Use .loc to avoid SettingWithCopyWarning when modifying a slice of the DataFrame
forecast_tomorrow.loc[:, 'ts'] = pd.to_datetime(forecast_tomorrow['ts'])  # Convert 'ts' in forecast_filtered to datetime

df_tomorrow_min = forecast_tomorrow.copy()
# Manually enter values for the last three rows of specified columns
df_tomorrow_min['Temperature_indoor'] = [13, 12, 11, 10, 10, 10, 10, 10, 12, 14, 15, 16, 17, 17, 17, 16.5, 16, 13, 12, 11, 11, "0", "0", "0" ]
df_tomorrow_min['door_indoors'] = 0.0
df_tomorrow_min['window_door'] = 1.0
df_tomorrow_min['curtain_right'] = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, "0", "0", "0" ]
df_tomorrow_min['curtain_left'] = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, "0", "0", "0" ]
df_tomorrow_min['shade_left'] = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, "0", "0", "0" ]
df_tomorrow_min['shade_right'] =  [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, "0", "0", "0" ]
df_tomorrow_min['pp1'] = 3.6
df_tomorrow_min['pp2'] = 26.2


columns_to_convert = ['ts','pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'Temperature_API', 'Temperature_indoor']
df_tomorrow_min[columns_to_convert] = df_tomorrow_min[columns_to_convert].astype('object')
df_tomorrow_min['ts'] = df_tomorrow_min['ts'].map(str) # make sure ts is a string type again (for json)


df_tomorrow_min.loc[df_tomorrow_min.index[-3:], 'door_indoors'] = ["0", "0", "0"]  # Replace value1, value2, value3 with actual values
df_tomorrow_min.loc[df_tomorrow_min.index[-3:], 'Temperature_API'] = ["0", "0", "0"]  # Replace value1, value2, value3 with actual values
df_tomorrow_min.loc[df_tomorrow_min.index[-3:], 'window_door'] = ["0", "0", "0"]  # Replace value1, value2, value3 with actual values

df_tomorrow_min = df_tomorrow_min.reindex(index=df_tomorrow_min.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
# Reset index and create a unique 'id' column
df_tomorrow_min.reset_index(inplace=True, drop=True)
df_tomorrow_min["id"] = df_tomorrow_min.index + 1
df_tomorrow_min['id'] = df_tomorrow_min['id'].map(str)

df_tomorrow_min

,ts,Temperature_API,Temperature_indoor,door_indoors,window_door,curtain_right,curtain_left,shade_left,shade_right,pp1,pp2,id
0,2024-10-05 23:00:00,0,0,0,0,0,0,0,0,3.6,26.2,1
1,2024-10-05 22:00:00,0,0,0,0,0,0,0,0,3.6,26.2,2
2,2024-10-05 21:00:00,0,0,0,0,0,0,0,0,3.6,26.2,3
3,2024-10-05 20:00:00,10.0,11,0.0,1.0,1.0,1.0,1.0,1.0,3.6,26.2,4
4,2024-10-05 19:00:00,10.0,11,0.0,1.0,1.0,1.0,1.0,1.0,3.6,26.2,5
5,2024-10-05 18:00:00,11.0,12,0.0,1.0,1.0,1.0,0.0,0.0,3.6,26.2,6
6,2024-10-05 17:00:00,12.0,13,0.0,1.0,1.0,0.0,0.0,0.0,3.6,26.2,7
7,2024-10-05 16:00:00,15.0,16,0.0,1.0,0.0,0.0,0.0,0.0,3.6,26.2,8
8,2024-10-05 15:00:00,16.0,16.5,0.0,1.0,0.0,0.0,0.0,0.0,3.6,26.2,9
9,2024-10-05 14:00:00,16.0,17,0.0,1.0,0.0,0.0,0.0,0.0,3.6,26.2,10


In [751]:
# Convert the string timestamps to datetime objects for comparison
start_time = pd.to_datetime('2024-10-05 00:00:00')
end_time = pd.to_datetime('2024-10-05 23:00:00')

# Filtering the forecast DataFrame to include only rows where the timestamp is between the two datetime objects
forecast_tomorrow = forecast[(forecast['ts'] >= start_time) & (forecast['ts'] <= end_time)]

# Use .loc to avoid SettingWithCopyWarning when modifying a slice of the DataFrame
forecast_tomorrow.loc[:, 'ts'] = pd.to_datetime(forecast_tomorrow['ts'])  # Convert 'ts' in forecast_filtered to datetime

df_tomorrow_max = forecast_tomorrow.copy()
# Manually enter values for the last three rows of specified columns
df_tomorrow_max['Temperature_indoor'] = [19, 18, 18, 17.5, 17.5, 17.5, 17.5, 17.5, 18, 19, 20, 21, 22, 22.5, 23, 22.5, 22, 21.5, 21, 20, 20, "0", "0", "0"  ]
df_tomorrow_max['door_indoors'] = 0.0
df_tomorrow_max['window_door'] = 0.0
df_tomorrow_max['curtain_right'] = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, "0", "0", "0" ]
df_tomorrow_max['curtain_left'] = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, "0", "0", "0" ]
df_tomorrow_max['shade_left'] = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, "0", "0", "0" ]
df_tomorrow_max['shade_right'] = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, "0", "0", "0" ]
df_tomorrow_max['pp1'] = 3.6
df_tomorrow_max['pp2'] = 26.2

columns_to_convert = ['ts','pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'Temperature_API', 'Temperature_indoor']
df_tomorrow_max[columns_to_convert] = df_tomorrow_max[columns_to_convert].astype('object')
df_tomorrow_max['ts'] = df_tomorrow_max['ts'].map(str) # make sure ts is a string type again (for json)


df_tomorrow_max.loc[df_tomorrow_max.index[-3:], 'door_indoors'] = ["0", "0", "0"]  # Replace value1, value2, value3 with actual values
df_tomorrow_max.loc[df_tomorrow_max.index[-3:], 'Temperature_API'] = ["0", "0", "0"]  # Replace value1, value2, value3 with actual values
df_tomorrow_max.loc[df_tomorrow_max.index[-3:], 'window_door'] = ["0", "0", "0"]  # Replace value1, value2, value3 with actual values

df_tomorrow_max = df_tomorrow_max.reindex(index=df_tomorrow_max.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
# Reset index and create a unique 'id' column
df_tomorrow_max.reset_index(inplace=True, drop=True)
df_tomorrow_max["id"] = df_tomorrow_max.index + 1
df_tomorrow_max['id'] = df_tomorrow_max['id'].map(str)

df_tomorrow_max

,ts,Temperature_API,Temperature_indoor,door_indoors,window_door,curtain_right,curtain_left,shade_left,shade_right,pp1,pp2,id
0,2024-10-05 23:00:00,0,0,0,0,0,0,0,0,3.6,26.2,1
1,2024-10-05 22:00:00,0,0,0,0,0,0,0,0,3.6,26.2,2
2,2024-10-05 21:00:00,0,0,0,0,0,0,0,0,3.6,26.2,3
3,2024-10-05 20:00:00,10.0,20,0.0,0.0,0.0,0.0,1.0,1.0,3.6,26.2,4
4,2024-10-05 19:00:00,10.0,20,0.0,0.0,0.0,0.0,1.0,1.0,3.6,26.2,5
5,2024-10-05 18:00:00,11.0,21,0.0,0.0,0.0,0.0,0.0,0.0,3.6,26.2,6
6,2024-10-05 17:00:00,12.0,21.5,0.0,0.0,0.0,1.0,0.0,0.0,3.6,26.2,7
7,2024-10-05 16:00:00,15.0,22,0.0,0.0,1.0,1.0,0.0,0.0,3.6,26.2,8
8,2024-10-05 15:00:00,16.0,22.5,0.0,0.0,1.0,1.0,0.0,0.0,3.6,26.2,9
9,2024-10-05 14:00:00,16.0,23,0.0,0.0,1.0,1.0,0.0,0.0,3.6,26.2,10


In [265]:
# def update_nan_times(df):
#     """
#     Update the 'NaN_times' column in the given DataFrame to indicate the indexes of NaN values
#     in specific columns and fill NaN values with 0.
    
#     Parameters:
#     df (DataFrame): The DataFrame to be processed.
#     """
#     df["NaN_times"] = "0"  # Create a column to store the notifications for NaN values
#     column_index = 0  # Initialize the index for updating the NaN_times column

#     # Iterate over the relevant columns to check for NaN values
#     for column in df[['window_door', 'curtain_left', 'curtain_right', 'shade_left', 'shade_right', 'door_indoors']]:
#         na_values = df[column].isna().tolist()  # Create a list indicating where NaN values are present
#         na_values = np.flip(na_values)
#         na_values = [i for i, n in enumerate(na_values) if n == True]  # List all indexes where NaN is True
#         # Check if there are any NaN values found
#         if any(na_values) == True:
#             # If NaN values are found, create a string indicating the first and last index of NaNs
#             na_values = [na_values[0]] + [na_values[-1]]  # Get the first and last index of NaN values
#             # na_values = 'na: ' + '-'.join(str(x) for x in na_values)  # Format the output string
#             na_values = ""  # If no NaN values, set to an empty string
#         else:
#             na_values = ""  # If no NaN values, set to an empty string

#         # Update the NaN_times column with the formatted NaN information for the current column
#         df.loc[column_index, 'NaN_times'] = na_values  
#         column_index += 1  # Increment the index for the next iteration

#     # Fill NaN values with 0 (since they will mess up the JSON output)
#     df.fillna("0", inplace=True)

# # Now apply the function to the three DataFrames
# update_nan_times(df_yesterday)
# update_nan_times(df_twodaysago)
# update_nan_times(df_threedaysago)
# update_nan_times(df_today)

# # Make sure again that all columns are of object dtype
# columns_to_convert = ['ts', 'date','pp1', 'pp2', 'curtain_left', 'door_indoors', 
#                       'shade_left', 'window_door', 'curtain_right', 'shade_right', 
#                       'time_since_today', 'Temperature_API', 'Temperature_indoor']
# df_yesterday[columns_to_convert] = df_yesterday[columns_to_convert].astype('object')
# df_twodaysago[columns_to_convert] = df_twodaysago[columns_to_convert].astype('object')
# df_threedaysago[columns_to_convert] = df_threedaysago[columns_to_convert].astype('object')
# df_today[columns_to_convert] = df_today[columns_to_convert].astype('object')
# df_today

In [329]:
df_today_max.dtypes

ts                    object
curtain_left          object
shade_left            object
curtain_right         object
shade_right           object
window_door           object
Temperature_indoor    object
door_indoors          object
date                  object
time_since_today      object
pp1                   object
pp2                   object
id                    object
Temperature_API       object
dtype: object

In [829]:
def post_data_to_api(df, url, api_token):
    """
    Function to post data to the specified API endpoint using the provided dataframe.
    
    Parameters:
    df (DataFrame): The dataframe containing the data to be sent.
    url (str): The API endpoint URL.
    api_token (str): The API token for authentication.
    """
    # Loop through the dataframe to add rows to the DF dataset as JSON
    for ind in df.index:
        # Create a ddictionary for headers to be sent to the API
        HEADERS = {
            'api_token': api_token,  # API token for authentication
            'resource_id': df["id"].iloc[ind],  # Resource ID from the dataframe
            'token': 'token_for_identifier'  # Token for additional authentication
        }
        
        # Extract parameters from the dataframe for the current index
        ts = df['ts'].iloc[ind]
        temp_out = df['Temperature_API'].iloc[ind]
        temp_in = df['Temperature_indoor'].iloc[ind]
        timesince = df['time_since_today'].iloc[ind]
        window_door = df['window_door'].iloc[ind]
        curtain_left = df['curtain_left'].iloc[ind]
        shade_left = df['shade_left'].iloc[ind]
        curtain_right = df['curtain_right'].iloc[ind]
        shade_right = df['shade_right'].iloc[ind]
        pp1 = df['pp1'].iloc[ind]
        pp2 = df['pp2'].iloc[ind]
        door_indoors = df['door_indoors'].iloc[ind]
        date_yesterday = df_yesterday['date'].iloc[ind]
        date_twodaysago = df_twodaysago['date'].iloc[0]
        date_threedaysago = df_threedaysago['date'].iloc[0]
        # NaN_times = df['NaN_times'].iloc[ind]
        
        # Prepare the parameters to be sent in the POST request
        PARAMS = {
            "pp1": pp1, 
            "pp2": pp2, 
            "door_indoors": door_indoors, 
            "window_door": window_door, 
            "curtain_left": curtain_left, 
            "shade_left": shade_left, 
            "curtain_right": curtain_right, 
            "shade_right": shade_right, 
            "Temperature_outdoor": temp_out, 
            "Temperature_indoor": temp_in, 
            "time_since_today": timesince, 
            "ts": ts,
            "date_yesterday": date_yesterday,
            "date_twodaysago": date_twodaysago,
            "date_threedaysago": date_threedaysago,
            # "NaN_times": NaN_times
        }    
        
        # Send a POST request to the API with the headers and parameters
        r = requests.post(url=url, headers=HEADERS, json=PARAMS)

# url_twodaysago = "https://data.id.tue.nl/datasets/entity/11744/item/"
# api_token_twodaysago = "NlBRYlJVWlRDS09LbHlDUlJpNXVqbjVuV1ZuV0hQSmNlemoxQkZsdXFBRT0="  

# url_threedaysago = "https://data.id.tue.nl/datasets/entity/11746/item/"
# api_token_threedaysago = "d1QwS3ZWSjBnR3U1RVREK3JtOEZLa0hkMitEY25GV3FBM3VkaEZ5Rm9uaz0="  

# url_yesterday = "https://data.id.tue.nl/datasets/entity/11745/item/"
# api_token_yesterday = "UzBNRUlmcE13ckJmRS9aSGxDdCszaXh2YTdrL0Q2QWhiUTFNMWhrd3g5bz0="    

url_today = "https://data.id.tue.nl/datasets/entity/11935/item/"
api_token_today = "NGpHTWhsU1BoQmRmZy9JekkxcGRuQnNock8ySTlBUkUxNk5YUUdxTjNlaz0="

url_today_max = "https://data.id.tue.nl/datasets/entity/11930/item/"
api_token_today_max = "V2JLS3FGamhwZDRCS3ZSVU96SlJXYVIxM3hhMVZORnNlSlMvRmVyZUFsdz0=" 

url_today_min = "https://data.id.tue.nl/datasets/entity/11929/item/"
api_token_today_min = "ZkRITHNheFFvR2VsUE1FY3pTeFVPcjljaG81WldkMW1sd1lwbTNRNmY0dz0=" 

# Call the function for each dataframe
# post_data_to_api(df_yesterday, url_yesterday, api_token_yesterday)
# post_data_to_api(df_twodaysago, url_twodaysago, api_token_twodaysago)
# post_data_to_api(df_threedaysago, url_threedaysago, api_token_threedaysago)
post_data_to_api(df_today, url_today, api_token_today)
post_data_to_api(df_today_max, url_today_max, api_token_today_max)
post_data_to_api(df_today_min, url_today_min, api_token_today_min)

In [691]:
def post_data_to_api(df, url, api_token):
    """
    Function to post data to the specified API endpoint using the provided dataframe.
    
    Parameters:
    df (DataFrame): The dataframe containing the data to be sent.
    url (str): The API endpoint URL.
    api_token (str): The API token for authentication.
    """
    # Loop through the dataframe to add rows to the DF dataset as JSON
    for ind in df.index:
        # Create a dictionary for headers to be sent to the API
        HEADERS = {
            'api_token': api_token,  # API token for authentication
            'resource_id': df["id"].iloc[ind],  # Resource ID from the dataframe
            'token': 'token_for_identifier'  # Token for additional authentication
        }
        
        # Extract parameters from the dataframe for the current index
        ts = df['ts'].iloc[ind]
        temp_out = df['Temperature_API'].iloc[ind]
        temp_in = df['Temperature_indoor'].iloc[ind]
        window_door = df['window_door'].iloc[ind]
        curtain_left = df['curtain_left'].iloc[ind]
        shade_left = df['shade_left'].iloc[ind]
        curtain_right = df['curtain_right'].iloc[ind]
        shade_right = df['shade_right'].iloc[ind]
        pp1 = df['pp1'].iloc[ind]
        pp2 = df['pp2'].iloc[ind]
        door_indoors = df['door_indoors'].iloc[ind]
        
        # Prepare the parameters to be sent in the POST request
        PARAMS = {
            "pp1": pp1, 
            "pp2": pp2, 
            "door_indoors": door_indoors, 
            "window_door": window_door, 
            "curtain_left": curtain_left, 
            "shade_left": shade_left, 
            "curtain_right": curtain_right, 
            "shade_right": shade_right, 
            "Temperature_outdoor": temp_out, 
            "Temperature_indoor": temp_in, 
            "ts": ts,
        }    
        
        # Send a POST request to the API with the headers and parameters
        r = requests.post(url=url, headers=HEADERS, json=PARAMS)

url_tomorrow_max = "https://data.id.tue.nl/datasets/entity/11936/item/"
api_tomorrow_max = "OGtWY0lrb2g1RElxWUUxSHlTME11Z2s1MFFidk1xMlZQUVlpRThXeE85ST0="  

url_tomorrow_min = "https://data.id.tue.nl/datasets/entity/11937/item/"
api_tomorrow_min = "RE1XODhCclVjSk9hbmZEYmZSdlFVbEovYVFOVGNndGIyaWZLMEszanpwTT0="  

# Call the function for each dataframe
post_data_to_api(df_tomorrow_max, url_tomorrow_max, api_tomorrow_max)
post_data_to_api(df_tomorrow_min, url_tomorrow_min, api_tomorrow_min)